# Week 6 · Lifecycles

[![Open in Colab](../assets/badges/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_06.ipynb)

An offer is not only data: it has a journey, from new to delivered. This week you write the rule that allows only the moves that make sense, and you see your rule drawn as a diagram.

| | |
|---|---|
| **This week's question** | Which moves may an offer make, and who decides? |
| **Python you will learn** | a dictionary of lists, `in`, `.get(key, default)`, counting into a dictionary, `while` loops, reading a class (the kit's `Offer`) |
| **Principle 6** | **Make wrong states hard to reach.** Allow only the moves that make sense; keep the rule next to the data. |
| **File you add** | `my_foodbridge/lifecycle.py` → `ALLOWED_MOVES` and `can_move(old_status, new_status)` |
| **What you will see** | `bridge.show("lifecycle", status="approved")`: the state diagram drawn from your `can_move`, with the current state lit and the allowed next moves in green |

*Campus FoodBridge is fictional. All organisations, people and numbers are invented.*

![System map, week 6: lifecycle.py allows only sensible moves](../assets/diagrams/system_map_w06.svg)

### By the end of this week you can …

- **explain** how a dictionary finds a value by its key, and ask safe questions with `in` and `.get`;
- **write** a `while` loop that is sure to end, and count things into a dictionary;
- **write** the offer's lifecycle as data (`ALLOWED_MOVES`) and its rule (`can_move`) in your own file, and **read** the class that uses it;
- **decide** whether a coordinator must approve before pickup, with a number from last semester as evidence.

### The story so far

Last week you built the door: `clean_offer` in `data_model.py` refuses bad rows with a reason.

This morning a clean offer caused a different problem. Lotus Cafe's curry (F004) was approved for the Old Market Elders' Group, but the screen already says **delivered**: a volunteer tapped the wrong button. Nobody has collected it, the elders are waiting, and the delivered total is wrong.

Clean data is not enough. The system must also guard each offer's journey. This week you add a fourth file, `lifecycle.py`.

## Before you start

1. **File → Save a copy in Drive.** Work in your copy.
2. Run the setup cell. In Colab it asks to connect Google Drive: say yes, so your `my_foodbridge/` folder is kept between weeks.
3. ⭐ = everyone · ⭐⭐ = stretch.

In [ ]:
# ▶ Run this cell first (setup)
# In Colab or Jupyter, this cell unpacks the FoodBridge kit next to the notebook.
# This web page is for reading. To run the notebook, use the 'Open in Colab' button at the top of this page.


## Class warm-up · Research 4 exchange (10 min)

For Research 4 you read how programs keep records in lists, dictionaries and objects (contacts, a bank account). Then you asked an AI about what stayed unclear, checked one claim in Colab, and wrote your view: *should a status be free text or one of a fixed list?* In groups of three:

1. Show the claim you checked with a tiny example. Did the book and the AI say the same thing? Was the AI right?
2. Free text or a fixed list? Each side gives its best reason. What happens to `"Deliverd"`, `"done"` or `"ok!!"` in each design?
3. This week the statuses become a fixed list in `ALLOWED_MOVES`. Which good point from the free-text side should the system still keep (for example, a note next to the status)?

📝 **My notes:** one idea from a classmate that I want to remember …

## Session 1 · Understand (90 min)

### The idea: entities, status and workflow

Think of a parcel you order online: *ordered → shipped → delivered*. The parcel is an **entity**: one real thing the system keeps track of. Its **status** says where it is on its journey. The **workflow** is the set of moves that make sense. A parcel is not delivered before it is shipped, and a delivered parcel does not go back to "ordered".

An offer has the same kind of journey:

![Offer lifecycle: new, approved, picked up, delivered, or rejected or cancelled](../assets/diagrams/lifecycle.svg)

People make these moves. A **coordinator** approves or rejects a new offer and chooses the partner; volunteers pick up and deliver. The program **records** the decision; whether the food is safe stays a human decision. In a diagram each status is called a **state**. Rejected, cancelled and delivered are **end states**: nothing moves out of them.

To write this journey in Python you need five tools: a dictionary of lists, `in`, `.get`, counting, and `while`.

### 1 · A dictionary finds a value by its key

📖 **What it is.** A dictionary keeps pairs: a **key** and its **value**. You give the key; Python gives back the value. In Week 3 each record was a dictionary (`offer["portions"]`). A value can also be a whole **list**: one day gives the list of dishes the canteen sells.

**Why it exists.** Finding a value by its name is quicker and clearer than searching through a list every time.

**How Python runs it.** Think of a row of drawers, each with a label. `MENU["tuesday"]` goes straight to the drawer labelled `"tuesday"`. **Each key is unique**: there is only one drawer with that label. If you store a second value under the same key, the new value **replaces** the old one. `in` asks a yes/no question. On a list it looks at the items; on a dictionary it looks at the **keys** only.

![MENU is a dictionary of lists: each day is a key, and its list holds that day&#x27;s dishes](../assets/diagrams/dict_of_lists.svg)

**Where you meet it.** Your phone's contacts (name → number), a price list (dish → price), the settings of an app, and the kit's table of Khmer messages.

| Code | The question it asks | Answer |
|---|---|---|
| `MENU["tuesday"]` | which value has the key `"tuesday"`? | the list of dishes |
| `"fruit" in MENU["tuesday"]` | is `"fruit"` one of the items in that list? | `True` or `False` |
| `"monday" in MENU` | is `"monday"` one of the **keys**? | `True` or `False` |

🤔 **Predict** the last two printed lines before you run the cell.

In [ ]:
MENU = {"monday": ["fried rice", "noodle soup"],
        "tuesday": ["fish amok", "fried rice", "fruit"]}
MENU["sunday"] = ["fruit"]          # a new key
MENU["sunday"] = []                 # the same key again: the new value replaces the old one
print(MENU["tuesday"])
print("fruit" in MENU["tuesday"], "fruit" in MENU)      # 🤔
print(len(MENU), MENU["sunday"])                        # 🤔

**Prediction:** …  **Result:** …  **Why:** …

### ✍️ Try it 1 · `sells`

Write `sells(day, dish)`: return `True` when the canteen sells that dish on that day. All the examples are given; run the cell until every line is ✅.

In [ ]:
def sells(day, dish):
    # TODO: look up the day's list in MENU, then ask with in
    return None

bridge.check("sells", sells, examples=[
    (("monday", "noodle soup"), True),
    (("monday", "fruit"), False),
    (("tuesday", "fruit"), True),
    (("sunday", "fried rice"), False),
])

🐞 **Common mistake · `KeyError: 'friday'`**

What happens if you ask your `sells` about a day that is not in `MENU`, such as `sells("friday", "fruit")`? The program stops:

```text
KeyError                         Traceback (most recent call last)
----> 1 sells("friday", "fruit")
      …
----> 2     return dish in MENU[day]

KeyError: 'friday'
```

**Read the error message** from the bottom up, as in Week 5. `KeyError` means "there is no drawer with this label", and `'friday'` is the label Python looked for. The arrow shows the line: `MENU[day]`. A typo does the same: `MENU["Monday"]` fails too, because `"Monday"` and `"monday"` are different keys. The fix is the next concept: a lookup that cannot crash.

### 2 · `.get(key, default)`: a lookup that cannot crash

📖 **What it is.** `MENU.get(key, default)` looks up the key, like `MENU[key]`. When the key is missing, it gives back the **default** you chose, instead of stopping. Without a default it gives `None`.

**Why it exists.** Real data has missing keys: a day the canteen is closed, a status with a typo, a field another app forgot. A program that stops at every missing key is fragile.

**How Python runs it.** If the drawer with that label exists, Python opens it. If not, it hands you the spare value you brought. Choose a default that means "nothing here": `[]` for a list ("no dishes", "nowhere to go"), `0` for a count.

**Where you meet it.** A chat app shows "no messages yet" for a new friend instead of crashing. A settings screen uses a default when you never chose one. The kit reads every setting this way.

| Code | The key exists | The key is missing |
|---|---|---|
| `MENU[day]` | the value | `KeyError`: the program stops |
| `MENU.get(day, [])` | the value | `[]` |
| `MENU.get(day)` | the value | `None` |

🤔 **Predict** the last two lines.

In [ ]:
print(MENU.get("monday", []))
print(MENU.get("friday", []))        # 🤔
print(MENU.get("friday"))            # 🤔 no default given

**Prediction:** …  **Result:** …  **Why:** …

### ✍️ Try it 2 · `dishes_on`

Write `dishes_on(day)`: return the list of dishes for that day, or `[]` for a day that is not in `MENU`.

This time one example is given, and you write two more. **Test the boundary:** the cases at the edge are where code breaks. Add one **boundary case** (a day that is in `MENU` but has no dishes) and one **odd case** (a day that is not in `MENU`, or `"Monday"` with a capital M). Write the expected answer first, then remove the `#`.

In [ ]:
def dishes_on(day):
    # TODO: one line with MENU.get and a default
    return None

bridge.check("dishes_on", dishes_on, examples=[
    ("monday", ["fried rice", "noodle soup"]),
    # ✍️ add your own: one boundary case and one odd case
    # ("...", [...]),
    # ("...", [...]),
])

### 3 · Counting into a dictionary

📖 **What it is.** A dictionary can hold a **count** for each value, such as `{"mon": 2, "tue": 1}`. You build it one row at a time, like tally marks on paper.

**Why it exists.** "How many of each?" is one of the most common questions an information system answers: offers per status, rejections per category, orders per day.

**How Python runs it.** For each row, take the key. Ask for its count so far with `.get(key, 0)`: the first time, there is none, so you get 0. Add 1, and store the new count under the same key. Because keys are unique, the new count replaces the old one.

```python
counts[key] = counts.get(key, 0) + 1
```

**Program thinking: trace by hand.** Before you trust a loop, follow it on paper, one round per line:

| Round | `key` | `counts.get(key, 0)` | `counts` after the round |
|---|---|---|---|
| 1 | `"mon"` | 0 | `{"mon": 1}` |
| 2 | `"tue"` | 0 | `{"mon": 1, "tue": 1}` |
| 3 | `"mon"` | 1 | `{"mon": 2, "tue": 1}` |

**Where you meet it.** A shop's sales per product, the votes in an election, the number of unread messages next to each chat. In Session 3 a count of last semester's offers is your evidence.

🤔 **Predict** the last printed line. Five students vote for lunch: trace the votes by hand, as in the table.

In [ ]:
votes = ["fruit", "rice", "fruit", "bread", "fruit"]
counts = {}
for vote in votes:
    counts[vote] = counts.get(vote, 0) + 1
    print(vote, counts)              # 🤔

**Prediction:** …  **Result:** …  **Why:** …

### ✍️ Try it 3 · `count_by`

Write `count_by(rows, field)`: return a dictionary `{value: how many rows have it}`. You will use it again in Weeks 8 and 10. Add your own cases: one boundary case (no rows at all) and one odd case (for example `"Mon"` and `"mon"` in the same list: one key or two?).

In [ ]:
def count_by(rows, field):
    counts = {}
    # TODO: for each row, take key = row[field], then add 1 to counts[key] with .get
    return counts

bridge.check("count_by", count_by, examples=[
    (([{"day": "mon"}, {"day": "tue"}, {"day": "mon"}], "day"), {"mon": 2, "tue": 1}),
    # ✍️ add your own: one boundary case and one odd case
    # (([...], "day"), {...}),
    # (([...], "day"), {...}),
])

### 👀 From counts to a picture

A dictionary of counts is ready to become a bar chart: the keys are the names of the bars, and the values are their heights. `counts.keys()` gives the bar names and `counts.values()` gives their heights; `list(...)` turns each into a list. `matplotlib` is a chart library that Colab already has. The cell uses your `count_by` on the four sample records, whose categories are bread, rice meals, fruit and rice meals. If the chart is empty, finish Try it 3 first. In Week 8 you draw your own charts and learn to make them honest.

![A dictionary of counts becomes a bar chart: keys are the bar names, values the heights](../assets/diagrams/counts_to_bar.svg)

🤔 **Predict** how many bars you will see, and which one is the tallest. Then run the cell and compare.

In [ ]:
import matplotlib.pyplot as plt

counts = count_by(bridge.sample_records(), "category")      # your Try it 3
plt.bar(list(counts.keys()), list(counts.values()))
plt.title("Sample offers by category")
plt.show()

🐞 **Common mistake · `counts[key] += 1` before the key exists**

A shorter line looks tempting:

```python
counts[key] += 1
```

It stops on the very first row:

```text
----> 4         counts[key] += 1

KeyError: 'mon'
```

`counts[key] += 1` means `counts[key] = counts[key] + 1`. Python first reads the old count, and the first time `"mon"` appears there is no old count, so there is no drawer to open. Use `.get` with the default 0: `counts[key] = counts.get(key, 0) + 1`.

### 4 · `while`: repeat as long as something is true

📖 **What it is.** A `while` loop repeats its block **as long as** a condition is `True`.

**Why it exists.** A `for` loop walks through a list you already have. Sometimes you do not know in advance how many rounds you need: keep paying while there is money, ask again until the input is valid, keep a menu open until the user types `q` (Week 9).

**How Python runs it.** `while` **checks the condition before every round**. When it is `True`, Python runs the block once, then goes back to the top and checks again. When it is `False`, Python skips the block and goes on below it. If the condition is `False` the first time, the block never runs. So something inside the block must move the condition towards `False`; if nothing does, the loop never ends.

```{iframe} https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_while_loop.html
:width: 100%
:title: while: the check before every round

[`while`: the check before every round](https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_while_loop.html)
```

**Where you meet it.** A game that runs until you quit, a phone that retries a download until it works, an ATM that asks for your PIN again (at most three times).

| Loop | Use it when … | It stops … |
|---|---|---|
| `for item in items:` | you have the list | after the last item |
| `while condition:` | you repeat until something changes | when the condition is `False` at the top |

🤔 **Predict** the printed line. Trace by hand: write the balance and the days after each round.

In [ ]:
balance = 5000              # riel of phone credit
days = 0
while balance >= 1500:      # one day of data costs 1500 riel; checked before every round
    balance = balance - 1500
    days = days + 1
print(days, balance)        # 🤔

**Prediction:** …  **Result:** …  **Why:** …

### ✍️ Try it 4 · `days_of_credit`

Write `days_of_credit(balance, daily_cost)`: return how many full days of data the credit pays for. Use a `while` loop. Add one boundary case (a balance that pays for **exactly** a whole number of days) and one odd case (a balance too small for even one day). Do not try a daily cost of 0 yet: read the box below first.

In [ ]:
def days_of_credit(balance, daily_cost):
    days = 0
    # TODO: while the balance pays for one more day, take the cost away and count the day
    return days

bridge.check("days_of_credit", days_of_credit, examples=[
    ((5000, 1500), 3),
    # ✍️ add your own: one boundary case and one odd case
    # ((..., ...), ...),
    # ((..., ...), ...),
])

🐞 **Common mistake · a loop that never ends**

What does `days_of_credit(5000, 0)` do? The balance never goes down, so `balance >= 0` stays `True` for ever. The cell keeps running: the run button keeps spinning and nothing is printed.

**How to stop it in Colab:** click the spinning stop button ■ next to the cell, or use **Runtime → Interrupt execution**. Python then prints a traceback that ends with:

```text
KeyboardInterrupt
```

That is not a new bug: it only says that *you* stopped the loop. **The fix:** make sure every round moves closer to the end. Here, check the input before the loop starts: when `daily_cost` is 0 or less, return 0 at once, because such a cost can never end the loop. When you are not sure a loop ends, add a safety limit, for example `while balance >= daily_cost and days < 365:`.

### 5 · The lifecycle as data

📖 **What it is.** A workflow can be written as **data**: a dictionary of lists. Each status is a key, and its list says where an offer may go next. An empty list means an **end state**: nowhere to go.

**Why it exists.** The rule then sits in one place that people can read, check and change. To change the workflow, you change the dictionary, not the code that uses it.

**How Python runs it.** "May an approved offer move to picked_up?" is two steps you already know: look up the key `"approved"`, then ask `in` its list. With `.get(status, [])`, an unknown status such as the typo `"deliverd"` gives "nowhere to go" instead of a `KeyError`.

```{iframe} https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_state_machine.html
:width: 100%
:title: The lifecycle as data: ALLOWED_MOVES and can_move

[The lifecycle as data: `ALLOWED_MOVES` and `can_move`](https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_state_machine.html)
```

**Where you meet it.** Parcel tracking, the order screen of a food-delivery app, a help-desk request (open → in progress → closed), a bank transfer that is pending, then done or refused.

| Question | Python |
|---|---|
| Where may an approved offer go next? | `ALLOWED_MOVES["approved"]` |
| May a new offer go straight to delivered? | `"delivered" in ALLOWED_MOVES["new"]` |
| Where may an unknown status go? | `ALLOWED_MOVES.get("deliverd", [])` → `[]` |

🤔 **Predict** the two printed lines. This is the lifecycle picture above, written as data.

In [ ]:
ALLOWED_MOVES = {
    "new": ["approved", "rejected"],
    "approved": ["picked_up", "cancelled"],
    "picked_up": ["delivered"],
    "delivered": [], "rejected": [], "cancelled": [],     # end states: nowhere to go
}
print(ALLOWED_MOVES["approved"])
print("delivered" in ALLOWED_MOVES["new"])                # 🤔

**Prediction:** …  **Result:** …  **Why:** …

👀 **A loop that must end.** This `while` loop follows one offer through the lifecycle. Before every round it asks: does the current status have any move left? If yes, it takes the first allowed move. It stops at an end state, a status whose list is empty.

🤔 **Predict** the printed lines. Then run the cell and compare.

In [ ]:
status = "new"
while ALLOWED_MOVES.get(status, []) != []:      # checked before every round: any move left?
    status = ALLOWED_MOVES[status][0]           # take the first allowed move
    print(status)                               # 🤔

The loop ends because every move goes forward and the lifecycle has end states. If someone added `"delivered": ["new"]`, it would go round for ever, like `days_of_credit(5000, 0)`. A lifecycle with no way back is easier to trust.

### ✍️ Try it 5 · `next_buttons` (FoodBridge)

A volunteer's phone should show only the buttons that make sense. Write `next_buttons(status)`: return the list of statuses an offer can move to, or `[]` for a status the lifecycle does not know. Add one boundary case (an end state) and one odd case (a status with a typo).

In [ ]:
def next_buttons(status):
    # TODO: one line with ALLOWED_MOVES.get
    return None

bridge.check("next_buttons", next_buttons, examples=[
    ("new", ["approved", "rejected"]),
    # ✍️ add your own: one boundary case and one odd case
    # ("...", [...]),
    # ("...", [...]),
])

### 🔁 Explain it back

Answer in your own words, without code.

1. You store a second value under a key that is already in a dictionary. What happens, and why?
2. The condition of a `while` loop never becomes `False`. What happens, and how do you stop it in Colab?
3. A person types the status. Why is `ALLOWED_MOVES.get(status, [])` safer than `ALLOWED_MOVES[status]`?

📝 **My answers:** 1. … 2. … 3. …

## Session 2 · Practise, then look under the hood (90 min)

*A new runtime since Session 1? Run the setup cell at the top again. This session needs nothing else from Session 1.*

### Part A · Practise

First you read the ticket and write the expected answers. Then you practise the skill three times, with less help each time: on paper, as a puzzle, and from a blank cell.

### 🎫 This week's ticket

> **From:** the coordinator · **To:** you
>
> On Tuesday a volunteer tapped "delivered" on Lotus Cafe's curry before anyone picked it up. On Wednesday someone moved a rejected offer back to approved. I approve offers and choose the partner. Record my decisions and the volunteers' steps, and refuse every other move, even a status with a typo.

| | |
|---|---|
| **Function** | `can_move(old_status, new_status)` in `my_foodbridge/lifecycle.py` |
| **Setting and data** | `REQUIRE_APPROVAL` (`True` or `False`) and `ALLOWED_MOVES` (a dictionary of lists), at the top of the file |
| **Input** | two statuses, as text |
| **Output** | `True` only for an allowed move · `False` for every other move, and for an unknown status (no `KeyError`) |
| **Done when** | `bridge.check("can_move", folder="my_foodbridge")` passes, and `bridge.show("lifecycle", status="approved")` lights `approved`, shows `picked_up` and `cancelled` as the next moves, and has no "missing?" arrows |

| | `old_status` | `new_status` | Output (with `REQUIRE_APPROVAL = True`) |
|---|---|---|---|
| 1 | `"new"` | `"approved"` | `True` |
| 2 | `"new"` | `"delivered"` | `False` |
| 3 | `"picked_up"` | `"delivered"` | ✍️ ? |
| 4 | `"new"` | `"picked_up"` | `False` |
| 5 | `"rejected"` | `"approved"` | `False` |
| 6 | `"deliverd"` | `"new"` | ✍️ ? |

📝 **Write the expected answer first**, before any code: row 3 → … because … · row 6 → … because …

<details><summary>Show the answer</summary>

Row 3 → `True`: a volunteer delivers food that was picked up. Row 6 → `False`: `"deliverd"` is a typo, a status the lifecycle does not know. The answer is "no", and the program must not stop with a `KeyError`.

</details>

### ✏️ Model it on paper first: the state diagram

Before you write code, draw the lifecycle on paper. Try not to look back at the picture from Session 1: draw it from what you know about the team.

1. Draw a box for each status: new, approved, picked_up, delivered, rejected, cancelled.
2. Draw an arrow for each allowed move. On the arrow, write **who** makes it: the coordinator or a volunteer.
3. Give each end state (no arrow out) a double border.
4. Check your drawing. Can an offer get stuck in a state that is not an end state? Is there any arrow back?
5. Under your drawing, write it as a dictionary of lists: one key per box, one list item per arrow. In Session 3 this becomes `ALLOWED_MOVES` in your file.

<details><summary>Show a possible answer</summary>

```python
{
    "new": ["approved", "rejected"],          # the coordinator decides
    "approved": ["picked_up", "cancelled"],   # a volunteer picks up, or the coordinator cancels
    "picked_up": ["delivered"],               # a volunteer delivers
    "delivered": [],
    "rejected": [],
    "cancelled": [],
}
```

There are three end states, and no arrow goes back: a mistake becomes a new move, such as `approved → cancelled`. What happens when a partner cannot take food that is already picked up? There is no arrow for that yet (⭐⭐ in Session 3).

</details>

### 🧩 Put the lines in order

A bus ticket follows `TICKET_MOVES`: booked → paid → used, and booked or paid → cancelled. Click the lines of `can_change` into order, then press **Check**. One line does not belong.

In [ ]:
bridge.parsons("ticket_moves")

*On the course website the puzzle does not run. Open the notebook in Colab, or write the letters in order in your notes.*

<details><summary>Show the answer</summary>

```python
def can_change(old_status, new_status):
    if old_status not in TICKET_MOVES:
        return False
    allowed = TICKET_MOVES[old_status]
    if new_status in allowed:
        return True
    return False
```

The line that does not belong is `allowed = TICKET_MOVES[new_status]`. It looks up the wrong status: the **old** status's list says where the ticket may go next.

Both styles are safe. You can also write the body in one line with `.get`: `return new_status in TICKET_MOVES.get(old_status, [])`. That is the style of the blank cell below and of your file.

</details>

### 🧩 From a blank cell

A phone top-up request starts as `requested`. It can be `paid` or `cancelled`. A paid request can be `topped_up` or `refunded`. `topped_up`, `cancelled` and `refunded` are end states. Write `TOPUP_MOVES` and `can_top_up(old_status, new_status)`. Two examples are given; add one boundary case (a move out of an end state) and one odd case (a status that does not exist).

In [ ]:
TOPUP_MOVES = {}

def can_top_up(old_status, new_status):
    return None

bridge.check("can_top_up", can_top_up, examples=[
    (("requested", "paid"), True),
    (("requested", "topped_up"), False),
    # ✍️ add your own: one boundary case and one odd case
    # (("...", "..."), ...),
    # (("...", "..."), ...),
])

### Part B · Under the hood

In this part you read code more than you write it. Follow along, predict, and write short notes.

### A class: data and the functions that work on it

📖 **What it is.** A **class** is a blueprint. It says which data each thing keeps (its **fields**) and which functions work on that data (its **methods**). An **object** is one thing made from the blueprint. Inside the kit, every offer is an object made from the class `Offer`.

**Why it exists.** Until now an offer was a plain dictionary, and the rules lived in separate functions. Any line of code could write `offer["status"] = "delivered"`, and nothing asked the rule. A class keeps `move_to` **next to the data**, so the right way is the easy way: the team changes an offer's status only through `move_to`, and `move_to` asks the rule first. (Python does not forbid `offer.status = "delivered"`; the team agrees not to write it.) That is this week's principle: make wrong states hard to reach.

**How Python runs it.** Calling the class, `Offer("F001", ...)`, makes a new object and runs `__init__` to fill in its fields. Inside a method, `self` means "this object". Each object keeps **its own** data: changing the bread's status does not change the curry's. `bread.move_to(...)` runs the method with `self` = the bread. Python fills in `self` for you, so you pass only the other arguments: with the kit's `Offer`, `offer.move_to("approved", can_move)` passes two. You can pass a function itself, such as your `can_move` without `()`, as an argument, just like you pass a function to `bridge.check`; inside `move_to` it is called `is_allowed`.

![The class Offer is a blueprint; the objects bread and curry keep their own data, and move_to changes only one object&#x27;s status](../assets/diagrams/class_objects.svg)

```{iframe} https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_class_object.html
:width: 100%
:title: A class makes objects: Offer, self and move_to

[A class makes objects: `Offer`, `self` and `move_to`](https://loem-ms.github.io/programming-for-information-systems-site/assets/anim/anim_class_object.html)
```

**Where you meet it.** Almost every real system has classes: a `User`, a `BankAccount` with `deposit` and `withdraw`, a `Parcel` with `ship`. AI assistants write classes often, so you need to read them. You will not write classes in this course.

Here is a shortened part of the kit's `Offer`:

```python
class Offer:
    """One food offer from a donor. Change its status only with move_to()."""

    def __init__(self, offer_id, donor, food, category, portions, hours_left, contact=""):
        self.id = offer_id
        self.donor = donor
        self.contact = contact
        self.food = food
        self.category = category
        self.portions = portions
        self.hours_left = hours_left
        self.status = "new"                 # every offer starts here
        self.partner = ""

    def move_to(self, new_status, is_allowed):
        """Change status only if is_allowed(old, new) is True. Return True or False."""
        if not is_allowed(self.status, new_status):
            return False                    # refused: nothing changes
        self.status = new_status
        return True
```

| Word | Meaning | In the kit's `Offer` |
|---|---|---|
| class | the blueprint | `Offer` |
| object | one thing made from the class | Sunrise Bakery's bread (F001), Lotus Cafe's curry (F004) |
| field | data each object keeps | `self.status`, `self.portions` |
| method | a function that belongs to the class | `move_to` |
| `self` | "this object" | the first parameter of every method |

🤔 **Read the real `Offer`** and answer before you open the answers:

1. What is the status of every new offer, and which line decides it?
2. What does `move_to` do when `is_allowed` returns `False`?

<details><summary>Show the answers</summary>

1. `"new"`: the line `self.status = "new"` in `__init__` runs for every new offer.
2. It returns `False` at once, before the line that changes the status, so **nothing changes**.

</details>

The first cell below makes a tiny copy of the class. It leaves out `is_allowed` and most fields, to show one thing only: two objects keep separate data. Run it: nothing is printed, because a blueprint is not yet a thing. Then 🤔 **predict** both statuses in the second cell, before you run it: two objects come from one class, and only the bread moves.

In [ ]:
class Offer:                                # a tiny copy of the kit's Offer
    def __init__(self, offer_id):
        self.id = offer_id                  # each object keeps its own data
        self.status = "new"
    def move_to(self, new_status):          # a method: a function inside the class
        self.status = new_status

In [ ]:
bread = Offer("F001")
curry = Offer("F004")
bread.move_to("approved")
print(bread.id, bread.status, "·", curry.id, curry.status)    # 🤔

**Prediction:** …  **Result:** …  **Why:** …

### 🧑‍🏫 How this week's file works

**Program thinking: decompose.** `lifecycle.py` splits the job into three small parts, each with one job: a setting (your decision), data (the diagram) and a rule (a function). The kit's `Offer` asks your rule before every move:

```text
a volunteer taps "picked up" on F004 (status: approved)
        │
        ▼
offer.move_to("picked_up", can_move)                  the kit's Offer object
        │  asks the rule
        ▼
can_move("approved", "picked_up")                     your lifecycle.py
        │  ALLOWED_MOVES.get("approved", [])  →  ["picked_up", "cancelled"]
        ▼
True  →  the status becomes "picked_up"       False  →  nothing changes; the screen says why
```

Here is `lifecycle.py` in outline (the full starter is in Session 3, *Write your file*):

```python
REQUIRE_APPROVAL = True                        # a setting: your decision

ALLOWED_MOVES = {                              # data: your state diagram
    "new": ["approved", "rejected"],
    # … one key for each status
}
if not REQUIRE_APPROVAL:                       # no approval needed? Then a new offer
    ALLOWED_MOVES["new"].append("picked_up")   # may also go straight to picked_up

def can_move(old_status, new_status):          # the rule: it answers True or False
    # … look up old_status with .get, then ask in
```

`append` adds `"picked_up"` to the list of `"new"`: one more arrow in the diagram, only when you switch approval off.

| Part of `lifecycle.py` | What it is | It receives | Value / gives back | Who uses it |
|---|---|---|---|---|
| `REQUIRE_APPROVAL` | a setting: your decision | — | `True` or `False` | the `if` below `ALLOWED_MOVES` |
| `ALLOWED_MOVES` | data: your state diagram | — | a dictionary of lists | `can_move` |
| `can_move(old_status, new_status)` | the rule | two statuses, as text | `True` or `False` | every move in the system, and `bridge.show("lifecycle")` |

Why does `can_move` only **answer**, instead of changing the status itself? One function, one job (Week 4): the rule says yes or no, and the offer changes itself. The same rule then works for every screen, and the check can test it with plain text.

🔭 **In real work.** Real systems often guard the status twice: in the code, as here, and in the database, which refuses to save a status that is not on its list of allowed values. Help-desk and project tools draw the same kind of state diagram from the same kind of data.

### 🤔 What happens if …

1. … a volunteer taps **delivered** on F004 while it is still `approved`?
2. … your `ALLOWED_MOVES` forgets the key `"picked_up"`?

📝 **My predictions:** 1. … 2. …

<details><summary>Show the answers</summary>

1. `move_to` asks `can_move("approved", "delivered")`. The list for `"approved"` is `["picked_up", "cancelled"]`, so the answer is `False`. `move_to` returns `False` and **nothing changes**: F004 stays approved, and the screen can say why.
2. `ALLOWED_MOVES.get("picked_up", [])` gives `[]`, so every picked-up offer is **stuck**: it can never be delivered. Nothing crashes, but `bridge.show("lifecycle")` draws a dotted "missing?" arrow from `picked_up` to `delivered`. With `ALLOWED_MOVES["picked_up"]` instead of `.get`, `can_move("picked_up", "delivered")` would stop with a `KeyError`, and `bridge.show` would add a 💥 note naming the error.

</details>

### ⚖️ Good or risky?

You asked an AI assistant: *"Volunteers need buttons to update an offer, and the coordinator wants a way to fix mistakes."* It wrote **Version A**. **Version B** is a teammate's rewrite. Both run. Read them in the cell below, then run it. Both try clicks on F004: Lotus Cafe's curry, 12 portions, approved for the Old Market Elders' Group.

In [ ]:
# Version A · from an AI assistant
def update_status(record, typed_status):
    record["status"] = typed_status.strip().lower()   # whatever the volunteer types

def mark_delivered(record):
    record["status"] = "delivered"                    # one click, from any status

def fix_mistake(record):
    record["status"] = "new"                          # reset: start again


# Version B · from a teammate
def move(record, new_status, is_allowed):
    old_status = record["status"]
    if is_allowed(old_status, new_status):
        record["status"] = new_status
        return "done: " + record["id"] + " " + old_status + " -> " + new_status
    return "refused: " + record["id"] + " cannot move from " + old_status + " to " + new_status


DEMO_MOVES = {
    "new": ["approved", "rejected"],
    "approved": ["picked_up", "cancelled"],
    "picked_up": ["delivered"],
    "delivered": [],
    "rejected": [],
    "cancelled": [],
}


def lifecycle_rule(old_status, new_status):
    return new_status in DEMO_MOVES.get(old_status, [])


curry_a = bridge.sample_records()[3]      # F004, approved
curry_b = bridge.sample_records()[3]
print("Version A")
mark_delivered(curry_a)
print("  1.", curry_a["id"], curry_a["status"])
fix_mistake(curry_a)
print("  2.", curry_a["id"], curry_a["status"])
update_status(curry_a, "Deliverd ")
print("  3.", curry_a["id"], curry_a["status"])
print("Version B")
print("  1.", move(curry_b, "delivered", lifecycle_rule))
print("  2.", move(curry_b, "new", lifecycle_rule))
print("  3.", move(curry_b, "picked_up", lifecycle_rule))
print("  4.", move(curry_b, "delivered", lifecycle_rule))

Use **Principle 6 (make wrong states hard to reach)**.

1. Find **three** problems in Version A. For each one, name the line, what happens to F004, and who is harmed.
2. Version B refuses two clicks. Is that a bug or a feature?
3. Version B works like the kit's `move_to`: one door that asks the rule. Can it stop someone from writing `record["status"] = "delivered"` somewhere else in the program?

📝 **My notes:** …

<details><summary>Instructor's view</summary>

1. **It skips steps.** `mark_delivered` works from any status. F004 becomes "delivered", but nobody picked it up: the Old Market Elders' Group waits for 12 portions that never come, and the delivered total grows by 12.
2. **It resets.** `fix_mistake` sends the curry back to "new". It may be approved again, a volunteer may be sent twice, and what really happened is lost.
3. **Free text becomes a status.** `"Deliverd "` becomes `"deliverd"`, a status no rule knows. F004 vanishes from the coordinator's list and from every total except "offered".
4. **No answer.** Version A returns nothing, so the screen cannot say whether the click worked.
5. **No single rule.** Three functions change the status in three ways; nobody can read or test the workflow in one place.

Version B has one door (`move`), one rule (`is_allowed`) and a clear answer. A refusal is a **feature**: nothing changed, and the screen says why. Mistakes become an allowed move, such as `approved → cancelled`, not a reset. Version B cannot stop `record["status"] = "delivered"` written elsewhere. A class with `move_to` makes the right way the easy way, but Python still cannot forbid the wrong way, so the team agrees that status changes only through one door, and code review checks it (Week 10).

</details>

## Session 3 · Build and decide (90 min)

*A new runtime? Run the setup cell at the top again. This session needs nothing else from earlier sessions.*

### 📁 Your folder now

In [ ]:
bridge.show()

### Write your file

Create `my_foodbridge/lifecycle.py` in the **Files** panel (right-click `my_foodbridge` → **New file**), paste the starter below, fill in the two `TODO`s, and save (**Ctrl+S** / **Cmd+S**) after every change.

- `ALLOWED_MOVES` is your paper diagram from Session 2, as a dictionary of lists.
- `REQUIRE_APPROVAL` is your decision; keep `True` for now. The `if` below the dictionary is done for you: without approval, a new offer may go straight to `picked_up`.
- `can_move` is the same pattern as `can_top_up`, with your dictionary.

```python
# lifecycle.py · Week 6: the moves an offer may make
REQUIRE_APPROVAL = True         # a coordinator approves before pickup

ALLOWED_MOVES = {
    "new": ["approved", "rejected"],
    # TODO 1: add the other five statuses from your paper diagram.
    #         An end state gets an empty list, for example "delivered": [],
}
if not REQUIRE_APPROVAL:
    ALLOWED_MOVES["new"].append("picked_up")


def can_move(old_status, new_status):
    """Return True when an offer may move from old_status to new_status."""
    # TODO 2: get the list for old_status with .get (default []),
    #         then return whether new_status is in that list
    return False
```

### ✅ Check it

**Saved** does not mean **correct**. The check asks eight questions: your six ticket rows, plus `approved → picked_up` and `delivered → new`. Row 4 depends on your setting: `can_move("new", "picked_up")` must be `False` while `REQUIRE_APPROVAL` is `True`.

In [ ]:
bridge.check("can_move", folder="my_foodbridge")

🤔 The unfinished starter says `False` to everything, yet it passes 5 of the 8 cases. Why is a rule that refuses everything "safe" but useless?

<details><summary>Show the answer</summary>

No wrong move gets through, but no right move either: every offer stays `new`, and no partner gets food. The three failing cases are the right moves.

</details>

### 👀 See it

The kit asks **your** `can_move` about every pair of statuses and draws an arrow for each `True`. `approved` is lit; the green states are the moves your rule allows next. A dotted "missing?" arrow is a move your rule forgets; a red dashed "unusual?" arrow is one it allows that should not happen.

In [ ]:
bridge.show("lifecycle", status="approved")

### 🧭 Your decision · must a coordinator approve before pickup?

Approval protects the partners: a person decides where the food goes, and can refuse it. But approval takes time, and fried rice with 2 hours left may spoil while it waits.

- `REQUIRE_APPROVAL = True` (the default): every offer waits for a person to decide.
- `REQUIRE_APPROVAL = False`: a new offer may go straight to `picked_up`. This is a **risky decision**: food leaves the donor before anyone has decided where it goes. If you choose it, say why, and how the team limits the risk.

The program only records decisions; it cannot judge whether food is safe. Before you decide, look at last semester. The cell below counts its 404 offers by status, with the same idea as your `count_by`. You only read the result.

In [ ]:
# @title Last semester's offers, counted by status { display-mode: "form" }
# Counts the offers in semester.csv (invented data) by status: all offers, and urgent ones.
import csv

with open("foodbridge_kit/data/semester.csv", encoding="utf-8") as file:
    semester = list(csv.DictReader(file))

all_offers, urgent_offers, rejected_by_category = {}, {}, {}
urgent_total = 0
for row in semester:
    status = row["status"]
    all_offers[status] = all_offers.get(status, 0) + 1
    if int(row["hours_left"]) <= 3:
        urgent_offers[status] = urgent_offers.get(status, 0) + 1
        urgent_total = urgent_total + 1
    if status == "rejected":
        rejected_by_category[row["category"]] = rejected_by_category.get(row["category"], 0) + 1

print("Last semester:", len(semester), "offers ·", urgent_total, "of them urgent (3 hours left or less)")
print()
print("status".ljust(12), "all offers".rjust(12), "urgent offers".rjust(15))
for status in all_offers:
    share_all = round(100 * all_offers[status] / len(semester))
    share_urgent = round(100 * urgent_offers.get(status, 0) / urgent_total)
    print(status.ljust(12), (str(all_offers[status]) + " (" + str(share_all) + "%)").rjust(12),
          (str(urgent_offers.get(status, 0)) + " (" + str(share_urgent) + "%)").rjust(15))
print()
print("Rejected offers by category:", rejected_by_category)

Read the table, then answer in your decision log:

1. How many offers did a coordinator **reject** last semester? What share of all offers is that? Without approval, these offers would have left the donor before anyone decided.
2. Compare the **cancelled** share of urgent offers with the cancelled share of all offers. What could waiting for approval have to do with it? (A count shows *what* happened, not *why*.)
3. 📌 Rice meals have the most rejections. Are rice meals the problem? Write your guess; Week 9 checks it.

Now decide. Open `my_foodbridge/lifecycle.py` in the **Files** panel, change `REQUIRE_APPROVAL` (or keep `True` and say why), save (**Ctrl+S** / **Cmd+S**), then run the cell below. The check still passes; with `False`, the diagram shows an orange "your decision" arrow from `new`.

In [ ]:
bridge.check("can_move", folder="my_foodbridge")
bridge.show("lifecycle", status="new")

**Decision log** — *My choice:* … · *An alternative:* … · *Why I chose it:* … · *Evidence (one number and where it came from):* … (e.g. *22 of 404 offers rejected by a coordinator, semester.csv, W6*) · *Who is affected / the tradeoff:* … · 📌 *My guess about rice meals:* …

### 📓 Project journal

In Week 10 you hand in your Final project with a short report (`README.md`). It explains what your system does and the decisions you made. Three lines each week now make that report easy to write later.

📝 *What my system does now:* … · *One decision I made (and why):* … · *One idea for my Final:* …

### ⭐⭐ Go further · a state of your own

Sometimes a partner cannot take food that a volunteer has already picked up. Your lifecycle has no arrow for that. Add a new state, `"returned"`: draw it on your paper diagram first, then add `"returned"` to the list of `"picked_up"` in `ALLOWED_MOVES` and give it its own key, `"returned": []` (an end state). Save (**Ctrl+S** / **Cmd+S**), run the check (it still passes), then show the lifecycle with `status="picked_up"`. The arrow to your new state appears in blue. A new rule like this is a first step towards a Final extension.

In [ ]:
# ⭐⭐ Show your new state: bridge.show("lifecycle", status="picked_up")

## Finish

- [ ] I shared my Research 4 view in the warm-up.
- [ ] My five ✍️ Try-it checks are ✅, including my own boundary and odd cases.
- [ ] I wrote the expected answers in the ticket before coding, and drew the state diagram on paper.
- [ ] `my_foodbridge/lifecycle.py` passes `bridge.check`, and `bridge.show("lifecycle")` has no "missing?" arrows.
- [ ] My decision log explains `REQUIRE_APPROVAL` with one number from last semester, and my project journal has three lines.

**Your system as a web app:** `bridge.start_web(folder="my_foodbridge")` (from Week 5) works at any time. The buttons on each offer now come from your `can_move`.

📚 **Read more:** [Python Basics](Python_Basics.ipynb) §05 Lists, dictionaries, tuples, sets · §06 Loops and adding things up · §13 Classes, objects, methods. One chapter: "Dictionaries" in [Python for Everybody](https://www.py4e.com/book) or "Dictionaries" in [Think Python](https://greenteapress.com/wp/think-python-3rd-edition/).

### Research 5 · What happens to data when the power goes off? (5%)

The question: what happens to a program's data when the power goes off? Learn about **files**, **file formats**, **databases** and **backups**. About one page.

1. **Read and organise.** Read [Python Basics](Python_Basics.ipynb) §11 Files and CSV and §12 JSON and JSON Lines. Then read **one** chapter: "Files" in [Python for Everybody](https://www.py4e.com/book), or "Reading and Writing Files" in [Automate the Boring Stuff](https://automatetheboringstuff.com/). Curious? Read the introduction of "Using Databases and SQL" in Python for Everybody too. Write a summary in your own words, in 5–8 sentences: what it is, why it exists, how it works, and one common mistake. Add one tiny example you ran, with its output.
2. **Ask an AI about what stayed unclear.** Ask at least two follow-up questions: one asks for an everyday example from Cambodia, and one asks *"what can go wrong?"*. Keep 6–10 key lines of the conversation.
3. **Compare and check.** Name one place where the book and the AI say the same thing in different ways, disagree, or where the AI was vague or wrong. Test one claim by running a tiny example in Colab. Was it right?
4. **Your view.** Write 150–250 words on *"Should a system ever really delete data?"* Take a side and give a reason from the FoodBridge case.
5. **Sources and honesty.** List the sections and chapters you read and the AI tool you used.

Use invented examples only, never real personal data. Bring your view to the **Week 7 class warm-up**. Submit by email to submit.progmis@gmail.com; the deadline and file name will be announced in class.

**Next week:** right now everything disappears when the runtime stops. You save the records to a file and keep a history of every move, in `my_foodbridge/storage.py`.

⬅ **Previous:** [Week 5 · Checking input](Week_05.ipynb) · **Next:** [Week 7 · Saving and history](Week_07.ipynb) ➡